Entire code by Arshnoor

In [ ]:
#import library
import requests
import json
import re
from datetime import datetime
from collections import Counter, defaultdict

In [ ]:
#API setup

#base url
BASE_URL = "https://data.sfgov.org/resource/g8m3-pdis.json"

#columns needed from the API
SELECT_FIELDS = [
    "dba_name",
    "full_business_address",
    "business_zip",
    "neighborhoods_analysis_boundaries",
    "location_start_date",
    "location_end_date",
    "naic_code",
    "naic_code_description",
    "business_corridor"
]

#max per API call
LIMIT = 1000

#filter for restaurant businesses (includes cafe, bars, fast food joint etc)
params_template = {
    "$select": ",".join(SELECT_FIELDS),
    "$where": "naic_code like '722%'",
    "$limit": LIMIT
}

In [ ]:

all_records = []
offset = 0

#pull rows
while True:
    params = params_template.copy()
    params["$offset"] = offset

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    batch = response.json()

    print(f"Fetched {len(batch)} rows at offset {offset}")

    all_records.extend(batch)
    #stops when rows are smaller than 1000
    if len(batch) < LIMIT:
        break

    offset += LIMIT

print(f"\nTotal records fetched: {len(all_records)}")


Fetched 1000 rows at offset 0
Fetched 1000 rows at offset 1000
Fetched 1000 rows at offset 2000
Fetched 1000 rows at offset 3000
Fetched 1000 rows at offset 4000
Fetched 1000 rows at offset 5000
Fetched 1000 rows at offset 6000
Fetched 1000 rows at offset 7000
Fetched 1000 rows at offset 8000
Fetched 1000 rows at offset 9000
Fetched 1000 rows at offset 10000
Fetched 1000 rows at offset 11000
Fetched 1000 rows at offset 12000
Fetched 1000 rows at offset 13000
Fetched 1000 rows at offset 14000
Fetched 1000 rows at offset 15000
Fetched 1000 rows at offset 16000
Fetched 150 rows at offset 17000

Total records fetched: 17150


In [ ]:
def parse_iso_date(date_str):
    #turn date text into python date
    if not date_str:
        return None
    try:
        return datetime.fromisoformat(date_str.split("T")[0])
    except Exception:
        return None

def compute_lifespan_days(start_dt, end_dt=None):
   #measure lifespan up to today for active restaurants
    if start_dt is None:
        return None
    if end_dt is None:
        end_dt = datetime.now()
    return (end_dt - start_dt).days


In [ ]:

def normalize_name(name):
    #clean names
    if not name:
        return None

    name = name.lower()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\b(inc|llc|corp|co|company|restaurant|restaurants)\b", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name if name else None


In [ ]:
#timestamp
ingested_at = datetime.now().isoformat()

cleaned_records = []

#start/end dates
#Get Active vs Closed restaurants
#Compute lifespan
#Add the normalized name
for record in all_records:
    start_dt = parse_iso_date(record.get("location_start_date"))
    end_dt = parse_iso_date(record.get("location_end_date"))

    status = "Active" if not record.get("location_end_date") else "Closed"
    lifespan_days = compute_lifespan_days(start_dt, end_dt)

    cleaned_record = {
        "dba_name": record.get("dba_name"),
        "normalized_name": normalize_name(record.get("dba_name")),
        "full_business_address": record.get("full_business_address"),
        "business_zip": record.get("business_zip"),
        "neighborhoods_analysis_boundaries": record.get("neighborhoods_analysis_boundaries"),
        "location_start_date": record.get("location_start_date"),
        "location_end_date": record.get("location_end_date"),
        "status": status,
        "lifespan_days": lifespan_days,
        "naic_code": record.get("naic_code"),
        "naic_code_description": record.get("naic_code_description"),
        "business_corridor": record.get("business_corridor"),
        "ingested_at": ingested_at
    }

    cleaned_records.append(cleaned_record)

print(f"Cleaned records created: {len(cleaned_records)}")

Cleaned records created: 17150


In [ ]:
# save to json
output_file = "sf_restaurants_clean.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(cleaned_records, f, ensure_ascii=False, indent=2)

print(f"Saved cleaned data to {output_file}")


Saved cleaned data to sf_restaurants_clean.json


In [ ]:

COVID_CUTOFF = datetime(2020, 1, 1)

post_covid_records = []

for record in cleaned_records:
    end_dt = parse_iso_date(record.get("location_end_date"))

# Keeps a record if
#   - still active (no end date)
#   - It closed on or after Jan 1, 2020
    if end_dt is None or end_dt >= COVID_CUTOFF:
        post_covid_records.append(record)

print(f"Post-COVID analysis cohort size: {len(post_covid_records)}")
print(f"Cutoff date used: {COVID_CUTOFF.date()}")

post_covid_output_file = "sf_restaurants_post_covid.json"

with open(post_covid_output_file, "w", encoding="utf-8") as f:
    json.dump(post_covid_records, f, ensure_ascii=False, indent=2)

print(f"Saved post-COVID cohort to {post_covid_output_file}")


Post-COVID analysis cohort size: 14460
Cutoff date used: 2020-01-01
Saved post-COVID cohort to sf_restaurants_post_covid.json


This analysis focuses on restaurants that were active at any point from 2020 onward, allowing us to study both pandemic-period closures and survival into the recovery period.

In [ ]:
#save to JSON
post_covid_output_file = "sf_restaurants_post_2020.json"

with open(post_covid_output_file, "w", encoding="utf-8") as f:
    json.dump(post_covid_records, f, ensure_ascii=False, indent=2)

print(f"Saved post-2020 cohort to {post_covid_output_file}")

Saved post-2020 cohort to sf_restaurants_post_2020.json


In [ ]:
#summary check
total_restaurants = len(post_covid_records)

status_counts = Counter(record["status"] for record in post_covid_records)

all_neighborhoods = [
    record["neighborhoods_analysis_boundaries"]
    for record in post_covid_records
    if record.get("neighborhoods_analysis_boundaries")
]

active_neighborhoods = [
    record["neighborhoods_analysis_boundaries"]
    for record in post_covid_records
    if record.get("neighborhoods_analysis_boundaries") and record.get("status") == "Active"
]

closed_neighborhoods = [
    record["neighborhoods_analysis_boundaries"]
    for record in post_covid_records
    if record.get("neighborhoods_analysis_boundaries") and record.get("status") == "Closed"
]

unique_neighborhoods = set(all_neighborhoods)

top_10_all = Counter(all_neighborhoods).most_common(10)
top_10_active = Counter(active_neighborhoods).most_common(10)
top_10_closed = Counter(closed_neighborhoods).most_common(10)

print("\nRESTAURANTS ACTIVE FROM 2020 ONWARD: SUMMARY CHECKS")
print("-" * 60)
print(f"Total restaurants in 2020+ cohort: {total_restaurants}")
print(f"Active restaurants: {status_counts.get('Active', 0)}")
print(f"Closed restaurants: {status_counts.get('Closed', 0)}")
print(f"Unique neighborhoods: {len(unique_neighborhoods)}")

print("\nTop 10 neighborhoods with the most restaurant businesses since 2020 (open + closed):")
for neighborhood, count in top_10_all:
    print(f"{neighborhood}: {count}")

print("\nTop 10 neighborhoods with the most restaurants still open today:")
for neighborhood, count in top_10_active:
    print(f"{neighborhood}: {count}")

print("\nTop 10 neighborhoods with the most restaurants that closed after 2020:")
for neighborhood, count in top_10_closed:
    print(f"{neighborhood}: {count}")


RESTAURANTS ACTIVE FROM 2020 ONWARD: SUMMARY CHECKS
------------------------------------------------------------
Total restaurants in 2020+ cohort: 14460
Active restaurants: 8293
Closed restaurants: 6167
Unique neighborhoods: 41

Top 10 neighborhoods with the most restaurant businesses since 2020 (open + closed):
Financial District/South Beach: 1546
Mission: 1395
Bayview Hunters Point: 932
South of Market: 847
Tenderloin: 711
Sunset/Parkside: 566
Mission Bay: 480
Chinatown: 460
Marina: 390
Nob Hill: 383

Top 10 neighborhoods with the most restaurants still open today:
Financial District/South Beach: 913
Mission: 757
Bayview Hunters Point: 479
Tenderloin: 419
South of Market: 398
Sunset/Parkside: 331
Mission Bay: 318
Chinatown: 309
Marina: 244
Nob Hill: 243

Top 10 neighborhoods with the most restaurants that closed after 2020:
Mission: 638
Financial District/South Beach: 633
Bayview Hunters Point: 453
South of Market: 449
Tenderloin: 292
Sunset/Parkside: 235
Mission Bay: 162
Chinatown

In [ ]:

#Compare neighborhood counts
all_counts = Counter(all_neighborhoods)
active_counts = Counter(active_neighborhoods)
closed_counts = Counter(closed_neighborhoods)

comparison_rows = []

comparison_neighborhoods = set(all_counts.keys()) | set(active_counts.keys()) | set(closed_counts.keys())

for neighborhood in comparison_neighborhoods:
    total_count = all_counts.get(neighborhood, 0)
    active_count = active_counts.get(neighborhood, 0)
    closed_count = closed_counts.get(neighborhood, 0)

    comparison_rows.append((neighborhood, total_count, active_count, closed_count))

comparison_rows = sorted(comparison_rows, key=lambda x: x[1], reverse=True)

print("\n" + "=" * 75)
print("NEIGHBORHOOD COMPARISON: TOTAL VS ACTIVE VS CLOSED")
print("=" * 75)
print(f"{'Neighborhood':35s} {'Total':>8s} {'Active':>8s} {'Closed':>8s}")
print("-" * 75)

for neighborhood, total_count, active_count, closed_count in comparison_rows:
    print(f"{neighborhood:35.35s} {total_count:8d} {active_count:8d} {closed_count:8d}")


NEIGHBORHOOD COMPARISON: TOTAL VS ACTIVE VS CLOSED
Neighborhood                           Total   Active   Closed
---------------------------------------------------------------------------
Financial District/South Beach          1546      913      633
Mission                                 1395      757      638
Bayview Hunters Point                    932      479      453
South of Market                          847      398      449
Tenderloin                               711      419      292
Sunset/Parkside                          566      331      235
Mission Bay                              480      318      162
Chinatown                                460      309      151
Marina                                   390      244      146
Nob Hill                                 383      243      140
Outer Richmond                           380      238      142
North Beach                              354      220      134
Inner Richmond                           343      207

This section compares neighborhoods by total restaurant presence since 2020, current active restaurants, and restaurants that have closed since 2020.

In [ ]:
#survival rates by neighborhood

post_covid_neighborhood_stats = defaultdict(lambda: {"total": 0, "active": 0, "closed": 0})

for record in post_covid_records:
    neighborhood = record.get("neighborhoods_analysis_boundaries")

    if not neighborhood:
        continue

    post_covid_neighborhood_stats[neighborhood]["total"] += 1

    if record["status"] == "Active":
        post_covid_neighborhood_stats[neighborhood]["active"] += 1
    else:
        post_covid_neighborhood_stats[neighborhood]["closed"] += 1

post_covid_survival_results = []

for neighborhood, stats in post_covid_neighborhood_stats.items():
    total = stats["total"]
    active = stats["active"]
    closed = stats["closed"]
    survival_rate = active / total if total > 0 else 0

    post_covid_survival_results.append({
        "neighborhood": neighborhood,
        "total_restaurants_active_since_2020": total,
        "currently_active": active,
        "closed_since_2020": closed,
        "survival_rate": round(survival_rate, 4)
    })

post_covid_survival_results = sorted(
    post_covid_survival_results,
    key=lambda x: x["survival_rate"],
    reverse=True
)

print("\nTOP NEIGHBORHOODS BY SURVIVAL RATE SINCE 2020")
print("-" * 70)

for row in post_covid_survival_results[:10]:
    print(
        f"{row['neighborhood']}: "
        f"Total={row['total_restaurants_active_since_2020']}, "
        f"Active={row['currently_active']}, "
        f"Closed={row['closed_since_2020']}, "
        f"Survival Rate={row['survival_rate'] * 100:.2f}%"
    )


TOP NEIGHBORHOODS BY SURVIVAL RATE SINCE 2020
----------------------------------------------------------------------
Lincoln Park: Total=1, Active=1, Closed=0, Survival Rate=100.00%
McLaren Park: Total=1, Active=1, Closed=0, Survival Rate=100.00%
Golden Gate Park: Total=13, Active=12, Closed=1, Survival Rate=92.31%
Presidio Heights: Total=68, Active=49, Closed=19, Survival Rate=72.06%
Haight Ashbury: Total=168, Active=118, Closed=50, Survival Rate=70.24%
Chinatown: Total=460, Active=309, Closed=151, Survival Rate=67.17%
Presidio: Total=24, Active=16, Closed=8, Survival Rate=66.67%
Mission Bay: Total=480, Active=318, Closed=162, Survival Rate=66.25%
Glen Park: Total=43, Active=28, Closed=15, Survival Rate=65.12%
Outer Mission: Total=137, Active=88, Closed=49, Survival Rate=64.23%
